# 02. VLM 프롬프트와 응답 검증 / 60분

**선행:** 01 규칙 기반 손 추종, 05.VLM 이미지 질문, Python JSON dict

**이 노트북의 결과:** 규칙 판단을 VLM으로 바꾸기 전에 프롬프트의 역할과 엄격한 JSON 응답을 확인합니다. `parse_hand_decision()`을 구현하고 형식 통과와 실제 손 상태 판단 정확도를 따로 기록합니다.

[단원 README](README.md) · [용어집](GLOSSARY.md)


<!-- lesson-purpose:start -->
## 이 실습은 무엇을 위한 것인가요?

거북이를 움직이기 전에 VLM이 손 이미지를 읽고 약속한 JSON으로 판단할 수 있는지 따로 확인합니다. 이 단계는 parser와 프롬프트의 검증이며 로봇 제어 실습이 아닙니다. 응답 형식과 실제 손 상태를 분리해 평가해야 다음 단계의 실패를 해석할 수 있습니다.

### 전체 과정에서 내 코드가 하는 일

```text
손 이미지·frame_id
  ↓
VLM 질문
  ↓
원문
  ↓
parser TODO
  ↓
형식 검사
  ↓
사람 기대값과 의미 비교·저장
```

command·hand_state·frame_id·reason의 정확한 schema와 값 관계를 검사합니다. 실제 관측 ID와의 대조 및 오래된 판단의 최종 STOP은 다음 03의 제어 책임입니다.

### 실행 전에 생각하고 결과로 설명하기

**질문:** 유효한 FOLLOW JSON이면 지금 거북이가 움직여야 하나요?

**설명:** 02에서는 움직이지 않습니다. JSON을 읽을 수 있다는 것과 비주먹이라는 의미 판단이 맞다는 것을 먼저 확인합니다. 03에서는 같은 프레임의 좌표·시각·pose까지 확인한 뒤에만 실제 속도를 결정합니다.

이후의 TODO는 각 절에서 지정한 위치만 수정하고, 정답 해설의 이유를 자기 입력에 적용해 설명합니다. 예제 실행 성공·학생 함수 검증·실제 장치 관찰은 해당 절에서 요구하는 범위대로 나누어 기록합니다.
<!-- lesson-purpose:end -->


## 실습 시작 확인

현재 노트북의 Python 커널과 `/workspace` 자료 경로를 확인합니다. 새 커널이거나 커널을 재시작했다면 첫 설정·import 셀부터 순서대로 실행합니다. 이전 노트북의 변수·import는 자동으로 전달되지 않습니다. 저장한 노트북·소스와 `Data` 파일은 유지됩니다. 접속·재부팅 복구는 [Setup 안내](../01.Setup/README.md)를 참고합니다.

**4일차 환경:** `nepes-course:physical-ai-v1.0`, Python `/opt/course/bin/python`을 사용합니다. 카메라·GUI는 로컬 화면에서 함께 관찰하고 입력 프로세스는 하나만 실행합니다.

**VLM 실습 준비:** VLM 질의에는 공용 Ollama의 `http://127.0.0.1:11434` API와 `gemma4:e2b-it-qat` 모델이 필요합니다. 해당 실습의 연결 확인에서 실패하면 로컬 Setup 담당자에게 서버·모델 준비를 요청합니다. 순수 Python fixture와 YOLO 실습에는 Ollama가 필요하지 않습니다.

**작업 순서:** 아래 첫 설정 셀의 `ROOT`가 `/workspace`인지 확인 → 현재 노트북이 지정한 `exercises/*.py`를 Jupyter 편집기에서 저장 → `student = load_file(...)` 검증 셀을 다시 실행합니다. 새 커널에는 이전 노트북의 변수나 import가 없으므로 이 노트북 자체의 설정·fixture 셀을 사용합니다. 실행 중 보이는 출력은 학습 확인용이고 배포본에는 저장하지 않습니다. TODO 실패와 카메라·모델 환경 실패를 구분하세요.

## 슬라이드와 함께 진행하기

시작 안내는 [4일차 슬라이드 18쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s054) · [배포 PDF 18쪽](../슬라이드/4일차.pdf#page=18)입니다. 노트북의 설명·TODO·검증·실제 실행을 진행한 뒤, [4일차 슬라이드 19쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s024) · [배포 PDF 19쪽](../슬라이드/4일차.pdf#page=19)로 돌아가 19–29쪽에서 결과를 복습합니다. 쪽수는 표지를 포함한 실제 페이지 순서입니다.


## 정답 활용 안내

정답 확인 시점과 직접 시도 시간은 강사가 현장 진도·이해도에 맞춰 조절합니다. 검증 통과 전에도 강사 안내에 따라 정답을 참고할 수 있습니다. 정답과 해설을 비교한 뒤 자신의 TODO에 반영하고 다시 검증하세요. 정답 실행만으로 자신의 TODO가 완성되지는 않습니다.


## 이번 실습이 전체 흐름에서 맡는 역할

[01 규칙 이동](01_ros_gesture_action.ipynb) → [02 VLM 프롬프트 검증](02_vlm_decision.ipynb) → [03 VLM 손 추종](03_ros_vlm_action.ipynb) → [04 판단 유지시간](04_ros_vlm_latency.ipynb) → [05 YOLO detection·pose](05_yolo_detection_pose.ipynb) → [06 pose 좌표·LLM·ROS](06_ros_pose_llm.ipynb). [07 segmentation](07_yolo_segmentation.ipynb)은 선택 자습입니다.

**구현 실습과 실제 실행의 관계:** 지정한 `exercises/` 파일에 학생이 알고리즘을 구현하고 노트북 검증 셀로 평가합니다. `solutions/`는 별도 참고 정답이며, 참고 정답 검증만 실행해도 학생 TODO가 완성되는 것은 아닙니다. 카메라·ROS 데모는 제공된 실행용 구현을 사용합니다. 학생 파일 저장이 실행 중인 ROS 노드를 자동으로 바꾸지는 않습니다. 실시간 동작과 자신의 함수 검증을 각각 완료하세요.


## 시간 지도

규칙→VLM 입출력 10분 → 프롬프트 10분 → parser TODO·검증 20분 → 실제 사진·오답 분석 15분 → 저장 5분


## 필요한 앞 내용 — 위치와 핵심을 함께 확인

[05.VLM/02_image_questions.ipynb의 이미지 질문 실습](../05.VLM/02_image_questions.ipynb)에서는 이미지 바이트를 base64로 넣어 Ollama `/api/chat`에 보내고 `message.content` 문자열을 읽습니다. 이 실습도 같은 방식이며 `gemma4:e2b-it-qat`, `stream=False`, 출력 schema를 지정합니다. 모델에게는 파일 이름으로 정답을 주지 않고 원본 손 이미지와 지시문을 줍니다.

[05.VLM/04_structured_results.ipynb의 구조화 응답 실습](../05.VLM/04_structured_results.ipynb)에 JSON 검사가 있습니다. 여기서는 문자열을 dict로 바꾼 뒤 네 key·문자열 타입·허용 상태를 검사합니다. 코드 fence나 JSON 뒤 설명을 임의 추출해 고치지 않고 실패로 처리합니다. 아래 셀은 `observation`, `decision`, `raw`를 직접 만들기 때문에 이전 파일/변수 없이 parser TODO를 검증할 수 있습니다. 실제 요청에는 `/workspace/Data/images/07.Mediapipe/person_open_palm.png`가 필요합니다.

In [ ]:
# 구분: 수업 핵심 셀
from pathlib import Path
import importlib.util, json, shutil, sys

def find_course_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "09.Pseudo-VLA").is_dir():
            return candidate
    raise RuntimeError("09.Pseudo-VLA를 포함한 과정 루트를 찾지 못했습니다.")

ROOT = find_course_root()
LESSON = ROOT / "09.Pseudo-VLA"
if str(LESSON) not in sys.path:
    sys.path.insert(0, str(LESSON))
OPTIONAL = {name: bool(importlib.util.find_spec(name)) for name in ('cv2', 'ultralytics', 'rclpy', 'requests')}
print({"course_root": str(ROOT), "optional_live_dependencies": OPTIONAL})

def load_file(module_name, relative_path):
    path = LESSON / relative_path
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 무엇을 모델에 맡길까?

<!-- procedure:lesson-31-7f8fee:start -->
![이동 전에 VLM 판단을 검증하기](../교재그림/절차/lesson-31.svg)

이 단계는 판단 검증입니다. JSON을 받았다고 거북이를 움직이지 않으며 실제 제어는 다음 실습입니다.
<!-- procedure:lesson-31-7f8fee:end -->


MediaPipe는 손 관절 좌표를 반환합니다. 손목과 손바닥 관절 0/5/9/13/17의 평균이 공통 목표 위치입니다.
VLM은 **같은 프레임의 원본 픽셀** 을 보고 주먹이면 STOP, 확실한 비주먹이면 FOLLOW를 제안합니다.
방향이나 속도는 요청하지 않으며 규칙 판정 결과를 프롬프트에 전달하지 않습니다.

| hand_state | 허용 command |
|---|---|
| NON_FIST | FOLLOW 또는 STOP |
| FIST / NO_HAND / UNKNOWN | STOP |

JSON은 `command, hand_state, frame_id, reason` 네 필드만 받습니다. bool/속도/설명/code fence는 거부합니다.
형식이 맞아도 주먹을 NON_FIST로 오판할 수 있습니다. 사람이 붙인 실제 정답과 별도로 비교해야 합니다.


In [ ]:
# 구분: 수업 핵심 셀
from common.hand_follow import rule_decision
observation = dict(frame_id="f1", captured_ms=1000, center=[1.0, 0.5], state="NON_FIST")
decision = rule_decision(observation)
pose = (5.5, 5.5, 0.0)
print(observation, decision, pose)
from common.hand_follow import hand_prompt
raw = json.dumps(decision)
print(hand_prompt("f1"))
print(raw)


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## TODO — [exercises/hand_decision.py](exercises/hand_decision.py)

`parse_hand_decision(raw)`를 작성합니다. `json.loads` 후 dict·정확한 key·문자열 타입·허용 값·FOLLOW와 상태의 일관성을 검사합니다.
성공하면 dict, 실패하면 None을 반환합니다. `object_pairs_hook`으로 중복 key도 거부합니다.
`reason`은 공백 제거 후 1–160자이고 frame_id는 비어 있지 않아야 합니다.
실제 관측과 frame_id가 같은지는 03의 `follow_motion`에서 검사하며 parser에 expected_frame_id 인자를 추가하지 않습니다.

<!-- todo-review:requirements:start -->
### 작성 전에 확인할 입력·출력과 순서

**입력 → 출력:** raw는 JSON 문자열입니다. `' {"command":"FOLLOW","hand_state":"NON_FIST","frame_id":"f1","reason":"open hand"} '`는 네 문자열의 dict를 반환하고, FOLLOW+FIST 또는 중복 command는 None입니다. reason/frame_id의 공백 제거 결과로 길이를 검사하되 성공 dict의 원문 값을 임의 수정하지 않습니다.
**작성 흐름 힌트:** object_pairs_hook에서 이미 본 key면 ValueError → json.loads 예외와 NaN 등 상수 거부 → 정확한 key 집합 → 모든 값 문자열 → 허용 command/state와 길이 → FOLLOW일 때 NON_FIST 조건 → 반환. 실패를 정상 FOLLOW 기본값으로 바꾸지 않습니다.
**설명할 지점:** “파싱이 성공했다는 것은 명세에 맞는 제안이라는 뜻입니다. 실제 영상과 frame ID가 같은지, 주먹을 잘못 본 것은 아닌지는 뒤 단계가 확인합니다.”
<!-- todo-review:requirements:end -->

### TODO만 읽고 작성할 수 있는 판정표

작성 위치는 `/workspace/09.Pseudo-VLA/exercises/hand_decision.py`의 `parse_hand_decision(raw)`입니다. 입력은 모델이 내놓은 JSON 문자열 하나, 출력은 정상 dict 또는 `None`입니다. 파일을 읽거나 Ollama를 호출하는 함수는 아닙니다. 아래 표와 순서대로 보조 함수와 parser를 작성하고 저장하세요.

| 검사 | 허용 조건 | 실패 처리 |
| --- | --- | --- |
| JSON 문법 | JSON object 하나, 동일 key가 두 번 없어야 함 | 파싱 예외·배열·code fence·추가 설명이면 None |
| key | command, hand_state, frame_id, reason 정확히 네 개 | 누락/추가 key이면 None |
| 값의 타입 | 네 값 모두 문자열 | 숫자·bool·null·list이면 None |
| command | FOLLOW 또는 STOP | 대소문자 변경/LEFT 등은 None |
| hand_state | NON_FIST, FIST, NO_HAND, UNKNOWN | 다른 값이면 None |
| frame_id | `.strip()` 후 비어 있지 않음 | 공백뿐이면 None |
| reason | `.strip()` 후 1~160자 | 공백뿐/161자 이상이면 None |
| 조합 | FOLLOW는 NON_FIST일 때만, STOP은 네 상태 모두 가능 | 모순이면 None |

`json.loads`의 object_pairs_hook는 dict로 합치기 **전** `(key,value)` 목록을 받습니다. 여기서 중복을 검사해야 뒤의 값이 앞의 STOP을 덮어쓰는 문제를 막습니다. `NaN`·`Infinity` 상수도 parse_constant로 거부합니다. 검사에만 strip을 쓰고 정상 반환값의 원문 공백은 유지합니다. JSON/타입 오류를 호출자에게 전파하지 않고 None으로 마칩니다.

**성공 기준:** 정상 FOLLOW/NON_FIST가 입력과 같은 dict로 반환되고, STOP/FIST도 정상이며 FOLLOW/FIST·중복 command·추가 speed·빈 frame_id·긴 reason이 모두 None입니다. 현재 관측의 frame_id와 실제로 같은지 확인하는 함수는 [03_ros_vlm_action.ipynb의 follow_motion TODO](03_ros_vlm_action.ipynb)이며, 여기서 필요한 관계는 `decision['frame_id']==observation['frame_id']`입니다. 이 parser에는 expected_frame_id 인자를 새로 추가하지 않습니다.


In [ ]:
# 구분: 수업 핵심 셀
student = load_file("student_hand_decision", "exercises/hand_decision.py")
assert student.parse_hand_decision(raw) == decision
assert student.parse_hand_decision(raw[:-1] + ', "command": "STOP"}') is None
for bad in ["```json\n{}\n```", "[]", json.dumps({**decision, "speed": 1}), json.dumps({**decision, "hand_state": "FIST"})]:
    assert student.parse_hand_decision(bad) is None
stop_decision = {**decision, "command": "STOP", "hand_state": "FIST"}
assert student.parse_hand_decision(json.dumps(stop_decision)) == stop_decision
for bad_value in (
    {**decision, "frame_id": "   "}, {**decision, "reason": " "},
    {**decision, "reason": "x" * 161}, {**decision, "command": True},
    {**decision, "hand_state": "OPEN"}, {**decision, "reason": None},
):
    assert student.parse_hand_decision(json.dumps(bad_value)) is None
spaced = {**decision, "frame_id": " f1 ", "reason": " open hand "}
assert student.parse_hand_decision(json.dumps(spaced)) == spaced
print("PASS: strict hand decision")


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 참고 정답

강사가 안내하는 시점에 아래 접힌 정답 셀을 여세요. 셀은 `solutions/`의 독립적인 참고 구현이 같은 계약을 통과하는지 확인합니다. 바로 아래 정답 해설·접힌 코드에 입력 검사·경계 처리·반환값을 다시 제공합니다. 실제 카메라·ROS 공통 구현은 별도 runtime이며 참고 정답 실행과 학생 TODO 검증은 각각 확인합니다.


In [ ]:
# 구분: 수업 핵심 셀
reference = load_file("reference_hand_decision", "solutions/hand_decision.py")
assert reference.parse_hand_decision(raw) == decision
assert reference.parse_hand_decision(raw[:-1] + ', "command": "STOP"}') is None
assert reference.parse_hand_decision(json.dumps({**decision, "command": "LEFT"})) is None
print("PASS reference")


### 정답 해설 — 입력부터 결과까지

**정답 구현 위치:** [solutions/hand_decision.py](solutions/hand_decision.py)의 `parse_hand_decision`입니다. JSON object를 읽을 때 중복 key를 거부하고 command·hand_state·frame_id·reason 네 필드만 허용합니다. 모든 값이 문자열인지, command/state가 허용 집합인지도 확인합니다.

FOLLOW는 NON_FIST와 함께일 때만 형식·상태 일관성을 만족합니다. FOLLOW+FIST는 거부합니다. code fence·배열·추가 speed 필드도 거부하여 파싱 실패는 None으로 돌려주고 controller가 STOP으로 처리하게 합니다.

NON_FIST라는 모델 답이 실제 손 상태와 맞는지는 parser가 알 수 없습니다. 형식 검증과 사람의 영상 정답 대조를 분리합니다. VLM에는 원본 이미지가 들어가며 MediaPipe의 규칙 판정을 정답 필터로 넣지 않습니다.

**확인 질문:** frame_id 필드가 존재하는 것과 현재 승인하려는 영상의 ID와 일치하는 것은 어느 단계에서 각각 확인하나요?

**적용·재검증:** 정답을 읽고 지정된 `exercises/` 파일의 TODO에 반영해 저장합니다. 노트북의 학습자 검증 셀을 다시 실행하고 경계 입력도 확인합니다. 참고 정답의 검증 통과는 학생 파일의 통과를 대신하지 않습니다. 이후 기존 실행·저장 셀에서 자신의 결과 경로를 확인합니다. 정답 알고리즘과 보조 함수는 표시된 solutions 파일에서 직접 읽습니다. 실제 카메라·ROS 실행은 제공 runtime을 사용하므로 학생 TODO 변경이 자동 반영된다고 가정하지 않습니다.

<!-- todo-review:answer:start -->
<details>
<summary>실행 없이 읽는 참고 정답과 단계별 주석</summary>

[실제 구현 파일](solutions/hand_decision.py)에서 가져온 코드입니다. 이 Markdown 블록은 실행되지 않습니다. 정답 알고리즘은 아래 파일 본문에 직접 구현되어 있습니다. 학생 파일에서는 기존 함수 이름과 인자를 유지하세요. 아래에 포함된 보조 함수·상수는 알고리즘을 읽기 위한 문맥입니다.

```python
"""참고 정답: JSON 중복 key·필수 필드·허용 상태·명령 모순을 검사합니다.
유효하면 dict, 손상되거나 모순된 판단이면 None을 반환합니다.

입력 검사부터 결과 반환까지 이 파일에서 직접 확인합니다.
실제 카메라·ROS 실행용 공통 구현과 분리한 학습용 정답입니다.
"""

from __future__ import annotations
import json

HAND_SCHEMA = {
    'type': 'object',
    'properties': {
        'command': {'type': 'string', 'enum': ['FOLLOW', 'STOP']},
        'hand_state': {'type': 'string', 'enum': ['FIST', 'NON_FIST', 'NO_HAND', 'UNKNOWN']},
        'frame_id': {'type': 'string', 'minLength': 1},
        'reason': {'type': 'string', 'minLength': 1, 'maxLength': 160},
    },
    'required': ['command', 'hand_state', 'frame_id', 'reason'],
    'additionalProperties': False,
}


def parse_hand_decision(raw):
    # dict로 변환된 뒤에는 중복 key가 사라지므로 JSON을 읽는 순간 검사합니다.
    def unique_object(pairs):
        value = {}
        for key, item in pairs:
            if key in value:
                raise ValueError(f'duplicate JSON key: {key}')
            value[key] = item
        return value

    try:
        value = json.loads(raw, object_pairs_hook=unique_object,
                           parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))
        if not isinstance(value, dict) or set(value) != set(HAND_SCHEMA['required']):
            return None
        if any(not isinstance(v, str) for v in value.values()):
            return None
        if value['command'] not in ('FOLLOW', 'STOP') or value['hand_state'] not in HAND_SCHEMA['properties']['hand_state']['enum']:
            return None
        if not value['frame_id'].strip() or not 1 <= len(value['reason'].strip()) <= 160:
            return None
        # 형식이 맞아도 FOLLOW와 주먹 상태가 모순되면 행동을 승인할 수 없습니다.
        if value['command'] == 'FOLLOW' and value['hand_state'] != 'NON_FIST':
            return None
        return value
    except (ValueError, TypeError):
        return None
```

</details>
<!-- todo-review:answer:end -->


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 실제 픽셀 요청

```bash
python /workspace/09.Pseudo-VLA/scripts/live_hand_decision.py --image /workspace/Data/images/07.Mediapipe/person_open_palm.png
```

직접 촬영한 주먹/비주먹/손 없음/가림 사진도 `--image`로 각각 실행합니다. 파일 이름을 정답으로 전달하지 않습니다.
VLM 원문·파싱 결과·지연이 Data에 저장됩니다. 모델 첫 로드가 느린 경우 준비 요청을 한 뒤 다시 측정합니다.
정지 이미지 시험의 180초 timeout은 로봇 제어 허용 시간과 다릅니다.

**실행 위치:** 위 명령은 JupyterLab의 Terminal(컨테이너 내부)에서 실행합니다. 기본 결과 파일은 `/workspace/Data/outputs/09.Pseudo-VLA/hand-decisions.jsonl`이며 한 시도당 `image,frame_id,model,raw,decision,stage,latency_ms,total_ms`가 한 JSON 행으로 추가됩니다. `stage`는 `preparation`(URL 준비), `model_show`(모델 조회), `image_read`(이미지 읽기), `inference`(요청·응답), `parse`(JSON 판정), `completed` 중 하나입니다. 오류 시 해당 단계와 `error,error_type`을 먼저 확인합니다. `latency_ms`는 추론 요청 이후 시간이며 요청 전 실패는 `null`, `total_ms`는 준비를 포함한 전체 시간입니다. `stage=parse`의 `decision=None`은 파싱 실패이고 `decision.hand_state`가 실제 손과 다른 것은 의미 판단 실패입니다. 둘을 다른 열에 기록하세요.

주먹·손 없음·가림 조건은 본인이 촬영한 파일을 `/workspace/Data/images/09.Pseudo-VLA/`에 저장하고 같은 `--image <절대경로>` 인자로 전달합니다. 원본을 준비하지 못한 조건은 미측정으로 둡니다. 네 조건의 입력 경로·기대 상태·실제 상태·schema 통과·잘못된 FOLLOW·불필요 STOP·latency_ms를 `/workspace/Data/outputs/course-review/09.Pseudo-VLA/`의 개인 기록에 저장합니다. 기존 jsonl은 추가 기록이므로 frame_id와 image를 확인해 이번 실행만 대조합니다.

**JSONL 자체가 없거나 새 행이 추가되지 않았다면:** 출력 디렉터리 권한·마운트·남은 공간을 확인하고 Terminal에 표시된 결과와 traceback을 보존합니다. 로그 저장에 실패해도 원래 요청 오류 traceback을 함께 출력합니다. 기존 실패 행은 지우지 않으며 재시도는 새 frame_id로 추가됩니다.


> **구분: 수업 핵심** — 표시된 시간 안에 강사와 함께 진행합니다.

## 완료 검사 / 평가

네 조건마다 expected state, actual state, schema 통과, 잘못된 FOLLOW, 불필요 STOP, latency_ms를 기록합니다.
parser 통과율과 장면 판단 정답률을 따로 계산하세요. confidence 숫자를 만들 필요가 없습니다.
실제 모델을 실행하지 않았다면 “미측정”으로 남기고 fixture 성공을 실제 인식 성공으로 쓰지 않습니다.

다음 [03 VLM 손 추종](03_ros_vlm_action.ipynb)에서는 이 응답을 같은 프레임의 손 좌표와 결합해 ROS 이동을 승인합니다. 여기서는 JSON을 읽고 검사하는 단계까지 수행하며 거북이를 실행하지 않습니다.




<!-- slide-sync:start -->
## 슬라이드로 돌아가기 — 결과 복습

[4일차 슬라이드 19쪽](https://docs.google.com/presentation/d/1M3huhBQOov8Du3WoRq0pTke0Aqfj33eLKloCLBtwOow/edit#slide=id.d4_s024) · [배포 PDF 19쪽](../슬라이드/4일차.pdf#page=19)에서 19–29쪽을 복습합니다. 자신의 TODO 검증과 실제 관측·미측정을 나누어 설명합니다.
<!-- slide-sync:end -->

> **구분: 수업 후 복습·자습** — 상세 원리와 확장 내용입니다.

## 자습
같은 사진에서 지시 문구만 바꾸어 결과를 비교합니다. 손 없음·여러 손·부분 가림에서 STOP이 유지되는지 확인하세요.
